# State Objective Fold-Local Target Generation v1

Goal: turn the fold-local BC readiness decision into an executable target-generation path. This notebook uses only train-window dates for each fold and then checks whether the resulting targets produce a healthier `risk_budget_usage_v1` BC label surface.

External basis:
- Qlib / qlib workflow discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL train-validation-test separation: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Backtest overfit control and walk-forward evidence discipline: https://doi.org/10.21314/JCF.2016.322

Important limitation: this is still audit-only. The target dates are fold-local, but the available teacher path comes from an existing selected OOS/router artifact. A promotion-grade run must regenerate the teacher path and state-objective targets inside each train fold.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.soxl_quant.config import load_profile
from src.soxl_quant.app.state_objective_training_targets import (
    build_fold_local_state_objective_training_targets,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
)
from src.soxl_quant.policy.policy_builder import build_state_objective_warm_start_dataset_from_frame

CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_fold_local_target_generation_v1"
TEACHER_CARD = ROOT / "runtime" / "reward" / "card" / "state_conditioned_rebound_risk_router_v1"
OBJECTIVE_CARD = ROOT / "runtime" / "model_selection" / "card" / "state_conditioned_objective_screen_v1"
FRAME_PATH = ROOT / "data" / "soxl_profile_balanced_rule_residual_v22.pkl"
CARD.mkdir(parents=True, exist_ok=True)

daily_paths = {
    "selected_strict_oos": pd.read_csv(TEACHER_CARD / "selected_strict_oos_daily.csv", parse_dates=["datetime"]),
}
cluster_metrics = pd.read_csv(OBJECTIVE_CARD / "cluster_metrics.csv")
feature_frame = pd.read_pickle(FRAME_PATH)
candidate = "state_conditioned_rebound_risk_router_v1"
daily_paths["selected_strict_oos"].head()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,datetime,price,buyhold_nav,strategy_nav,base_position,upside_position,router_position,rebound_state,risk_state,mom_5,mom_21,drawdown_63,rv_21,vol_of_vol_21,soxl_soxx_ratio_widen_21,soxl_soxx_ratio_z_252,soxx_mom_21
0,2020-01-02,19.464001,1000000.000000,1000000.000000,0.0625,0.205,0.0625,False,False,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,2020-01-03,18.412666,945985.699589,996624.106224,0.0625,0.205,0.0625,False,False,0.0,0.0,-0.054014,0.0,0.0,0.0,0.0,0.0
2,2020-01-06,17.828667,915981.609334,994648.468561,0.0625,0.205,0.0625,False,False,0.0,0.0,-0.084018,0.0,0.0,0.0,0.0,0.0
3,2020-01-07,18.747999,963214.062639,997854.024687,0.0625,0.205,0.0625,False,False,0.0,0.0,-0.036786,0.0,0.0,0.0,0.0,0.0
4,2020-01-08,18.789333,965337.683206,997991.524193,0.0625,0.205,0.0625,False,False,0.0,0.0,-0.034662,0.0,0.0,0.0,0.0,0.0


## Fold Specs

`wf_2020` cannot be generated from this teacher source because there is no pre-2020 teacher train path in the selected artifact. The two folds below are date-filtered train windows only.

In [2]:
folds = [
    {
        "name": "wf_2022_train",
        "source_window": "selected_strict_oos",
        "train_start": "2020-01-02",
        "train_end": "2021-12-31",
        "eval_start": "2022-01-03",
        "eval_end": "2022-12-30",
    },
    {
        "name": "wf_2023_2024_train",
        "source_window": "selected_strict_oos",
        "train_start": "2020-01-02",
        "train_end": "2022-12-30",
        "eval_start": "2023-01-03",
        "eval_end": "2024-12-31",
    },
]
folds

[{'name': 'wf_2022_train',
  'source_window': 'selected_strict_oos',
  'train_start': '2020-01-02',
  'train_end': '2021-12-31',
  'eval_start': '2022-01-03',
  'eval_end': '2022-12-30'},
 {'name': 'wf_2023_2024_train',
  'source_window': 'selected_strict_oos',
  'train_start': '2020-01-02',
  'train_end': '2022-12-30',
  'eval_start': '2023-01-03',
  'eval_end': '2024-12-31'}]

## Build Fold-Local Audit Targets

The reusable builder rejects overlapping train/eval windows and filters clusters to the train window. The current external cluster file contains 2025/2026 rebound clusters, so no rebound clusters survive into these train folds; this is a limitation of the available source artifact, not a permission to train.

In [3]:
fold_targets = build_fold_local_state_objective_training_targets(
    daily_paths,
    folds=folds,
    cluster_metrics=cluster_metrics,
    candidate=candidate,
)
fold_targets.to_csv(CARD / "fold_local_audit_targets.csv", index=False)
fold_target_summary = (
    fold_targets.groupby("fold", dropna=False)
    .agg(
        rows=("datetime", "count"),
        unique_dates=("datetime", "nunique"),
        rebound_rows=("is_rebound_cluster", "sum"),
        downtrend_rows=("is_downtrend_state", "sum"),
        mean_teacher_position=("teacher_position", "mean"),
        mean_risk_target_position=("risk_target_position", "mean"),
        max_objective_weight=("objective_weight", "max"),
    )
    .reset_index()
)
fold_target_summary.to_csv(CARD / "fold_target_summary.csv", index=False)
fold_target_summary

,fold,rows,unique_dates,rebound_rows,downtrend_rows,mean_teacher_position,mean_risk_target_position,max_objective_weight
0,wf_2022_train,505,505,0,234,0.207103,0.193649,226.658515
1,wf_2023_2024_train,756,756,0,478,0.161971,0.152684,1.000000


## BC Label Surface Check

The same fold-local targets are mapped through candidate action surfaces. The target dates are fold-local, so this tests action-surface suitability. It still does not test promotion-grade source provenance.

In [4]:
profiles = ["risk_budget_usage_v1", "balanced", "balanced_rule_residual_v22"]
label_rows = []
fold_label_rows = []
for profile_name in profiles:
    profile = load_profile(profile_name)
    obs, actions = build_state_objective_warm_start_dataset_from_frame(
        feature_frame,
        targets=fold_targets,
        profile=profile,
        sample_size=0,
    )
    dist = pd.Series(actions).value_counts(normalize=True).sort_index()
    label_rows.append(
        {
            "profile": profile_name,
            "samples": int(len(actions)),
            "obs_dim": int(obs.shape[1]),
            "unique_actions": int(dist.shape[0]),
            "dominant_action": int(dist.idxmax()),
            "dominant_action_share": float(dist.max()),
            "action_entropy": float(-(dist * np.log(dist + 1e-12)).sum()),
        }
    )
    for fold_name, subset in fold_targets.groupby("fold"):
        fold_obs, fold_actions = build_state_objective_warm_start_dataset_from_frame(
            feature_frame,
            targets=subset,
            profile=profile,
            sample_size=0,
        )
        fold_dist = pd.Series(fold_actions).value_counts(normalize=True).sort_index()
        fold_label_rows.append(
            {
                "fold": fold_name,
                "profile": profile_name,
                "samples": int(len(fold_actions)),
                "obs_dim": int(fold_obs.shape[1]),
                "unique_actions": int(fold_dist.shape[0]),
                "dominant_action": int(fold_dist.idxmax()),
                "dominant_action_share": float(fold_dist.max()),
                "action_entropy": float(-(fold_dist * np.log(fold_dist + 1e-12)).sum()),
            }
        )

label_comparison = pd.DataFrame(label_rows).sort_values(["dominant_action_share", "action_entropy"], ascending=[True, False])
fold_label_comparison = pd.DataFrame(fold_label_rows).sort_values(["fold", "dominant_action_share"])
label_comparison.to_csv(CARD / "profile_label_comparison.csv", index=False)
fold_label_comparison.to_csv(CARD / "fold_profile_label_comparison.csv", index=False)
readiness_scores = score_state_objective_bc_readiness(label_comparison, target_scope="fold_local")
readiness_summary = summarize_state_objective_bc_readiness(readiness_scores)
readiness_scores.to_csv(CARD / "bc_readiness_scores.csv", index=False)
label_comparison

,profile,samples,obs_dim,unique_actions,dominant_action,dominant_action_share,action_entropy
0,risk_budget_usage_v1,755,315,5,1,0.396026,1.202464
1,balanced,755,79,9,0,0.903311,0.457479
2,balanced_rule_residual_v22,755,340,5,0,0.964238,0.197395


## Decision

The fold-local target builder is usable and keeps eval rows out. The audit target table again points to `risk_budget_usage_v1` as the least degenerate action surface. Training still remains blocked until the teacher path / cluster targets are regenerated inside each train fold, especially because the current cluster source contributes no train-fold rebound clusters.

In [5]:
selection = {
    "experiment": "state_objective_fold_local_target_generation_v1",
    "decision": "fold_local_audit_targets_generated_no_training",
    "promotion_allowed": False,
    "training_allowed": False,
    "selected_profile_for_next_training": readiness_summary["selected_profile"],
    "readiness_decision_if_source_is_fold_native": readiness_summary["decision"],
    "source_provenance_blocker": "teacher daily path and cluster metrics were selected outside these train folds; regenerate them inside each fold before training",
    "cluster_blocker": "available rebound clusters are outside the train folds, so high-vol rebound participation targets are absent in this audit table",
    "next_mechanism": "regenerate_fold_native_teacher_paths_and_clusters_then_budget_usage_bc_warm_start_ppo",
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": 30.0,
        "max_dd_max_pct": 15.0,
        "walkforward_required": True,
        "fold_native_teacher_source_required": True,
        "label_distribution_gate_required": True,
    },
    "sidecars": [
        "fold_local_audit_targets.csv",
        "fold_target_summary.csv",
        "profile_label_comparison.csv",
        "fold_profile_label_comparison.csv",
        "bc_readiness_scores.csv",
        "summary.json",
    ],
}
summary = {
    "selection": selection,
    "fold_target_summary": fold_target_summary.to_dict(orient="records"),
    "label_comparison": label_comparison.to_dict(orient="records"),
    "readiness_summary": readiness_summary,
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True))
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True))
selection

{'experiment': 'state_objective_fold_local_target_generation_v1',
 'decision': 'fold_local_audit_targets_generated_no_training',
 'promotion_allowed': False,
 'training_allowed': False,
 'selected_profile_for_next_training': 'risk_budget_usage_v1',
 'readiness_decision_if_source_is_fold_native': 'state_objective_bc_ready_for_training',
 'source_provenance_blocker': 'teacher daily path and cluster metrics were selected outside these train folds; regenerate them inside each fold before training',
 'cluster_blocker': 'available rebound clusters are outside the train folds, so high-vol rebound participation targets are absent in this audit table',
 'next_mechanism': 'regenerate_fold_native_teacher_paths_and_clusters_then_budget_usage_bc_warm_start_ppo',
 'acceptance_gate_for_next_run': {'relative_return_min_pp': 30.0,
  'max_dd_max_pct': 15.0,
  'walkforward_required': True,
  'fold_native_teacher_source_required': True,
  'label_distribution_gate_required': True},
 'sidecars': ['fold_loca